In [2]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline

def chronological_lap_split(
    df,
    lap_col="lap_number",
    train_end_lap=32,
    test_start_lap=33,
    test_end_lap=None,
):
    if test_end_lap is None:
        test_end_lap = int(df[lap_col].max())

    train = df.loc[df[lap_col] <= train_end_lap].copy()

    test = df.loc[
        df[lap_col].between(test_start_lap, test_end_lap)
    ].copy()

    if train.empty or test.empty:
        raise ValueError("Chronological split produced an empty partition.")

    assert train[lap_col].max() < test[lap_col].min()

    print(
        f"Train laps: {train[lap_col].min()}-{train[lap_col].max()}, "
        f"rows={len(train)}"
    )
    print(
        f"Test laps: {test[lap_col].min()}-{test[lap_col].max()}, "
        f"rows={len(test)}"
    )

    return train, test

In [3]:
def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        return {
            "n": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "Bias": np.nan,
            "R2": np.nan,
        }

    return {
        "n": len(y_true),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "Bias": float(np.mean(y_pred - y_true)),
        "R2": (
            r2_score(y_true, y_pred)
            if len(y_true) > 1 else np.nan
        ),
    }


def evaluate_regressors(
    train,
    test,
    feature_cols,
    target_col="lap_duration",
):
    missing = [
        c for c in feature_cols + [target_col]
        if c not in train.columns or c not in test.columns
    ]

    if missing:
        raise KeyError(f"Missing model columns: {sorted(set(missing))}")

    train = train.dropna(subset=[target_col]).copy()
    test = test.dropna(subset=[target_col]).copy()

    X_train = train[feature_cols].copy()
    X_test = test[feature_cols].copy()

    # Convert categorical features before this step or use a dedicated
    # categorical encoder. This implementation expects numeric columns.
    non_numeric = [
        c for c in feature_cols
        if not pd.api.types.is_numeric_dtype(X_train[c])
    ]
    if non_numeric:
        raise TypeError(
            f"Encode categorical columns first: {non_numeric}"
        )

    y_train = train[target_col].astype(float)
    y_test = test[target_col].astype(float)

    models = {
        "Training median": DummyRegressor(strategy="median"),
        "Random forest": RandomForestRegressor(
            n_estimators=400,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
        ),
        "Histogram gradient boosting": HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=42,
        ),
    }

    rows = []
    fitted_models = {}

    for name, estimator in models.items():
        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("model", estimator),
        ])

        model.fit(X_train, y_train)
        prediction = model.predict(X_test)

        rows.append({
            "model": name,
            **calculate_metrics(y_test, prediction),
        })

        fitted_models[name] = model

    # Baseline 2: use each driver's latest completed lap before the test
    # window. This assumes the test set starts after the training window.
    if "driver_number" in train.columns and "driver_number" in test.columns:
        last_lap_by_driver = (
            train.sort_values("lap_number")
                 .groupby("driver_number")["lap_duration"]
                 .last()
        )

        baseline_pred = test["driver_number"].map(last_lap_by_driver)

        if baseline_pred.notna().any():
            baseline_mask = baseline_pred.notna()
            rows.append({
                "model": "Last observed driver lap",
                **calculate_metrics(
                    test.loc[baseline_mask, target_col],
                    baseline_pred.loc[baseline_mask],
                ),
            })

    return (
        pd.DataFrame(rows).sort_values("MAE"),
        fitted_models,
    )

In [4]:
def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    valid = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        return {
            "n": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "Bias": np.nan,
            "R2": np.nan,
        }

    return {
        "n": len(y_true),
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": mean_squared_error(y_true, y_pred) ** 0.5,
        "Bias": float(np.mean(y_pred - y_true)),
        "R2": (
            r2_score(y_true, y_pred)
            if len(y_true) > 1 else np.nan
        ),
    }


def evaluate_regressors(
    train,
    test,
    feature_cols,
    target_col="lap_duration",
):
    missing = [
        c for c in feature_cols + [target_col]
        if c not in train.columns or c not in test.columns
    ]

    if missing:
        raise KeyError(f"Missing model columns: {sorted(set(missing))}")

    train = train.dropna(subset=[target_col]).copy()
    test = test.dropna(subset=[target_col]).copy()

    X_train = train[feature_cols].copy()
    X_test = test[feature_cols].copy()

    # Convert categorical features before this step or use a dedicated
    # categorical encoder. This implementation expects numeric columns.
    non_numeric = [
        c for c in feature_cols
        if not pd.api.types.is_numeric_dtype(X_train[c])
    ]
    if non_numeric:
        raise TypeError(
            f"Encode categorical columns first: {non_numeric}"
        )

    y_train = train[target_col].astype(float)
    y_test = test[target_col].astype(float)

    models = {
        "Training median": DummyRegressor(strategy="median"),
        "Random forest": RandomForestRegressor(
            n_estimators=400,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1,
        ),
        "Histogram gradient boosting": HistGradientBoostingRegressor(
            max_iter=250,
            learning_rate=0.05,
            max_leaf_nodes=15,
            l2_regularization=1.0,
            random_state=42,
        ),
    }

    rows = []
    fitted_models = {}

    for name, estimator in models.items():
        model = Pipeline([
            ("imputer", SimpleImputer(strategy="median", add_indicator=True)),
            ("model", estimator),
        ])

        model.fit(X_train, y_train)
        prediction = model.predict(X_test)

        rows.append({
            "model": name,
            **calculate_metrics(y_test, prediction),
        })

        fitted_models[name] = model

    # Baseline 2: use each driver's latest completed lap before the test
    # window. This assumes the test set starts after the training window.
    if "driver_number" in train.columns and "driver_number" in test.columns:
        last_lap_by_driver = (
            train.sort_values("lap_number")
                 .groupby("driver_number")["lap_duration"]
                 .last()
        )

        baseline_pred = test["driver_number"].map(last_lap_by_driver)

        if baseline_pred.notna().any():
            baseline_mask = baseline_pred.notna()
            rows.append({
                "model": "Last observed driver lap",
                **calculate_metrics(
                    test.loc[baseline_mask, target_col],
                    baseline_pred.loc[baseline_mask],
                ),
            })

    return (
        pd.DataFrame(rows).sort_values("MAE"),
        fitted_models,
    )

In [5]:
def rolling_origin_evaluation(
    df,
    feature_cols,
    target_col="lap_duration",
    lap_col="lap_number",
    folds=None,
):
    if folds is None:
        folds = [
            (15, 16, 24),
            (24, 25, 32),
            (32, 33, 40),
            (40, 41, 46),
            (46, 47, 52),
        ]

    results = []

    for train_end, test_start, test_end in folds:
        train, test = chronological_lap_split(
            df,
            lap_col=lap_col,
            train_end_lap=train_end,
            test_start_lap=test_start,
            test_end_lap=test_end,
        )

        metrics, _ = evaluate_regressors(
            train=train,
            test=test,
            feature_cols=feature_cols,
            target_col=target_col,
        )

        metrics["train_end_lap"] = train_end
        metrics["test_start_lap"] = test_start
        metrics["test_end_lap"] = test_end

        results.append(metrics)

    return pd.concat(results, ignore_index=True)

In [6]:
def compound_weather_diagnostic(
    df,
    pace_col="lap_duration",
    compound_col="compound",
    weather_col="weather_state",
    tyre_age_col="tyre_age",
):
    required = [pace_col, compound_col, weather_col]
    missing = [c for c in required if c not in df.columns]

    if missing:
        raise KeyError(f"Missing diagnostic columns: {missing}")

    group_cols = [weather_col, compound_col]

    if tyre_age_col in df.columns:
        df = df.copy()
        df["tyre_age_band"] = pd.cut(
            df[tyre_age_col],
            bins=[-1, 5, 10, 20, 30, 1000],
            labels=["0-5", "6-10", "11-20", "21-30", "31+"],
        )
        group_cols.append("tyre_age_band")

    summary = (
        df.groupby(group_cols, dropna=False, observed=False)[pace_col]
          .agg(["count", "median", "mean", "std"])
          .reset_index()
          .sort_values(group_cols)
    )

    return summary

# Example:
# display(compound_weather_diagnostic(lap_df))

In [10]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SESSION_DIR = PROJECT_ROOT / "data" / "processed" / "9558"
REPAIR_DIR = PROJECT_ROOT / "data" / "repaired"

print("Session directory exists:", SESSION_DIR.exists())
print("\nProcessed session files:")

for path in sorted(SESSION_DIR.rglob("*")):
    if path.is_file():
        print(path.relative_to(PROJECT_ROOT))

print("\nRepaired files:")
if REPAIR_DIR.exists():
    for path in sorted(REPAIR_DIR.rglob("*")):
        if path.is_file():
            print(path.relative_to(PROJECT_ROOT))

Session directory exists: True

Processed session files:
data\processed\9558\car_data\1.parquet
data\processed\9558\car_data\10.parquet
data\processed\9558\car_data\11.parquet
data\processed\9558\car_data\14.parquet
data\processed\9558\car_data\16.parquet
data\processed\9558\car_data\18.parquet
data\processed\9558\car_data\2.parquet
data\processed\9558\car_data\20.parquet
data\processed\9558\car_data\22.parquet
data\processed\9558\car_data\23.parquet
data\processed\9558\car_data\24.parquet
data\processed\9558\car_data\27.parquet
data\processed\9558\car_data\3.parquet
data\processed\9558\car_data\31.parquet
data\processed\9558\car_data\4.parquet
data\processed\9558\car_data\44.parquet
data\processed\9558\car_data\55.parquet
data\processed\9558\car_data\63.parquet
data\processed\9558\car_data\77.parquet
data\processed\9558\car_data\81.parquet
data\processed\9558\fuel_features.parquet
data\processed\9558\historical_pit_events.parquet
data\processed\9558\lap_performance.parquet
data\proces

In [11]:

from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SESSION_DIR = PROJECT_ROOT / "data" / "processed" / "9558"

# Reuse the chronological evaluation outputs already generated.
result_files = {
    "Fold results": SESSION_DIR / "rolling_temporal_fold_results.csv",
    "Compound results": SESSION_DIR / "rolling_temporal_compound_results.csv",
    "Weather results": SESSION_DIR / "rolling_temporal_weather_results.csv",
    "Weather-compound results": (
        SESSION_DIR / "rolling_temporal_weather_compound_results.csv"
    ),
    "Predictions": SESSION_DIR / "rolling_temporal_predictions.parquet",
}

existing_results = {}

for name, path in result_files.items():
    print(f"\n{'=' * 65}")
    print(f"{name}: {path.name}")

    if not path.exists():
        print("FILE NOT FOUND")
        continue

    if path.suffix == ".csv":
        df = pd.read_csv(path)
    else:
        df = pd.read_parquet(path)

    existing_results[name] = df

    print("Shape:", df.shape)
    print("Columns:", df.columns.tolist())

    if name == "Predictions":
        print("\nSample predictions:")
        display(df.head(5))
    else:
        display(df.head(10))

print("\nInspection complete. No files were changed.")



Fold results: rolling_temporal_fold_results.csv
Shape: (25, 6)
Columns: ['fold', 'model', 'n', 'MAE', 'RMSE', 'Bias']


,fold,model,n,MAE,RMSE,Bias
0,16-24,Linear Regression,166.0,4.850503,8.605221,-4.741857
1,16-24,Random Forest,166.0,4.334377,5.619903,-1.789469
2,16-24,Ridge Regression,166.0,5.450937,7.330593,-0.833174
3,16-24,Training median baseline,166.0,3.958235,6.010076,-3.641199
4,16-24,XGBoost,166.0,3.831159,5.938427,-3.690535
5,25-32,Linear Regression,133.0,5.090953,6.728745,-2.202110
6,25-32,Random Forest,133.0,3.435931,4.810108,0.026091
7,25-32,Ridge Regression,133.0,4.265073,5.887806,-1.703096
8,25-32,Training median baseline,133.0,10.003917,10.692961,-10.003917
9,25-32,XGBoost,133.0,4.267221,5.841704,-0.134608



Compound results: rolling_temporal_compound_results.csv
Shape: (20, 6)
Columns: ['model', 'compound', 'n', 'MAE', 'RMSE', 'Bias']


,model,compound,n,MAE,RMSE,Bias
0,Linear Regression,HARD,27.0,4.675679,7.618031,-4.322367
1,Linear Regression,INTERMEDIATE,211.0,2.748515,3.733552,0.084556
2,Linear Regression,MEDIUM,258.0,5.228553,8.601777,-4.750134
3,Linear Regression,SOFT,135.0,5.008748,9.525753,-4.065574
4,Random Forest,HARD,27.0,2.788257,3.765347,1.628436
5,Random Forest,INTERMEDIATE,211.0,3.026001,4.008492,1.532533
6,Random Forest,MEDIUM,258.0,3.707446,5.095730,-0.756868
7,Random Forest,SOFT,135.0,2.775686,3.600559,2.076577
8,Ridge Regression,HARD,27.0,3.797725,5.781367,-3.610274
9,Ridge Regression,INTERMEDIATE,211.0,2.281227,2.901960,0.483413



Weather results: rolling_temporal_weather_results.csv
Shape: (10, 6)
Columns: ['model', 'weather_regime', 'n', 'MAE', 'RMSE', 'Bias']


...


Weather-compound results: rolling_temporal_weather_compound_results.csv
Shape: (115, 8)
Columns: ['fold', 'model', 'weather_regime', 'compound', 'n', 'MAE', 'RMSE', 'Bias']


,fold,model,weather_regime,compound,n,MAE,RMSE,Bias
0,16-24,Linear Regression,No rain,HARD,3,2.209010,2.252260,-2.209010
1,16-24,Linear Regression,No rain,MEDIUM,50,2.069609,2.470958,-2.067974
2,16-24,Linear Regression,No rain,SOFT,2,3.599993,3.759508,-3.599993
3,16-24,Linear Regression,Rain,HARD,1,20.983942,20.983942,-20.983942
4,16-24,Linear Regression,Rain,INTERMEDIATE,13,9.162493,10.237132,-9.085176
5,16-24,Linear Regression,Rain,MEDIUM,95,5.512729,10.123867,-5.334325
6,16-24,Linear Regression,Rain,SOFT,2,12.035208,14.212443,-12.035208
7,16-24,Random Forest,No rain,HARD,3,2.181624,2.229554,-2.181624
8,16-24,Random Forest,No rain,MEDIUM,50,2.093984,2.464788,-2.093984
9,16-24,Random Forest,No rain,SOFT,2,3.041488,3.245239,-3.041488



Predictions: rolling_temporal_predictions.parquet
Shape: (3155, 12)
Columns: ['fold', 'model', 'row_idx', 'actual', 'predicted', 'lap_number', 'compound', 'rainfall', 'error', 'abs_error', 'squared_error', 'weather_regime']

Sample predictions:


,fold,model,row_idx,actual,predicted,lap_number,compound,rainfall,error,abs_error,squared_error,weather_regime
0,16-24,Training median baseline,15,92.504,92.697,16,MEDIUM,0.0,0.193,0.193,0.037249,No rain
1,16-24,Training median baseline,16,95.359,92.697,17,MEDIUM,0.0,-2.662,2.662,7.086244,No rain
2,16-24,Training median baseline,17,96.172,92.697,18,MEDIUM,0.0,-3.475,3.475,12.075625,No rain
3,16-24,Training median baseline,18,102.485,92.697,19,MEDIUM,1.0,-9.788,9.788,95.804944,Rain
4,16-24,Training median baseline,19,97.108,92.697,20,MEDIUM,1.0,-4.411,4.411,19.456921,Rain



Inspection complete. No files were changed.


In [12]:

import numpy as np
import pandas as pd

predictions = existing_results["Predictions"].copy()

# Overall performance across all chronological test predictions
overall = (
    predictions.groupby("model")
    .agg(
        n=("error", "count"),
        MAE=("abs_error", "mean"),
        mean_squared_error=("squared_error", "mean"),
        Bias=("error", "mean"),
    )
    .reset_index()
)

overall["RMSE"] = np.sqrt(overall["mean_squared_error"])
overall = overall.drop(columns="mean_squared_error")
overall = overall.sort_values("MAE").reset_index(drop=True)

print("OVERALL CHRONOLOGICAL MODEL RANKING")
display(overall.round(3))

# Compare performance separately in each temporal fold
fold_summary = (
    predictions.groupby(["fold", "model"])
    .agg(
        n=("error", "count"),
        MAE=("abs_error", "mean"),
        mean_squared_error=("squared_error", "mean"),
        Bias=("error", "mean"),
    )
    .reset_index()
)

fold_summary["RMSE"] = np.sqrt(fold_summary["mean_squared_error"])
fold_summary = fold_summary.drop(columns="mean_squared_error")

print("\nPERFORMANCE BY CHRONOLOGICAL FOLD")
display(
    fold_summary.sort_values(["fold", "MAE"]).round(3)
)

# Check that the saved metrics agree with the prediction-level calculations
print("\nPREDICTION INTEGRITY CHECK")
print("Prediction rows:", len(predictions))
print("Missing errors:", predictions["error"].isna().sum())
print(
    "Maximum error reconstruction difference:",
    (
        predictions["error"]
        - (predictions["actual"] - predictions["predicted"])
    ).abs().max()
)


OVERALL CHRONOLOGICAL MODEL RANKING


,model,n,MAE,Bias,RMSE
0,XGBoost,631,3.191,0.046,4.694
1,Random Forest,631,3.241,0.717,4.401
2,Ridge Regression,631,3.955,-1.528,6.062
3,Linear Regression,631,4.329,-2.969,7.537
4,Training median baseline,631,5.848,-3.399,7.247



PERFORMANCE BY CHRONOLOGICAL FOLD


,fold,model,n,MAE,Bias,RMSE
4,16-24,XGBoost,166,3.831,-3.691,5.938
3,16-24,Training median baseline,166,3.958,-3.641,6.010
1,16-24,Random Forest,166,4.334,-1.789,5.620
0,16-24,Linear Regression,166,4.851,-4.742,8.605
2,16-24,Ridge Regression,166,5.451,-0.833,7.331
6,25-32,Random Forest,133,3.436,0.026,4.810
7,25-32,Ridge Regression,133,4.265,-1.703,5.888
9,25-32,XGBoost,133,4.267,-0.135,5.842
5,25-32,Linear Regression,133,5.091,-2.202,6.729
8,25-32,Training median baseline,133,10.004,-10.004,10.693



PREDICTION INTEGRITY CHECK
Prediction rows: 3155
Missing errors: 0
Maximum error reconstruction difference: 64.94591454244724


In [13]:

import numpy as np
import pandas as pd

predictions = existing_results["Predictions"].copy()

# Reconstruct both possible error conventions.
actual_minus_predicted = predictions["actual"] - predictions["predicted"]
predicted_minus_actual = predictions["predicted"] - predictions["actual"]

print("ERROR CONVENTION CHECK")
print(
    "Max difference vs (actual - predicted):",
    (predictions["error"] - actual_minus_predicted).abs().max()
)
print(
    "Max difference vs (predicted - actual):",
    (predictions["error"] - predicted_minus_actual).abs().max()
)

print("\nDERIVED-COLUMN CHECKS")
print(
    "Max abs_error mismatch:",
    (predictions["abs_error"] - predictions["error"].abs()).abs().max()
)
print(
    "Max squared_error mismatch:",
    (
        predictions["squared_error"] - predictions["error"] ** 2
    ).abs().max()
)

# Show rows where the saved error matches neither convention.
tol = 1e-6
mismatch = predictions[
    (predictions["error"] - actual_minus_predicted).abs().gt(tol)
    & (predictions["error"] - predicted_minus_actual).abs().gt(tol)
]

print("\nROWS MATCHING NEITHER ERROR CONVENTION")
print("Count:", len(mismatch))
display(
    mismatch[
        [
            "fold", "model", "row_idx", "actual", "predicted",
            "error", "abs_error", "squared_error"
        ]
    ].head(15)
)

# Recalculate metrics directly from actual and predicted.
predictions["recomputed_error"] = (
    predictions["actual"] - predictions["predicted"]
)
predictions["recomputed_abs_error"] = predictions["recomputed_error"].abs()
predictions["recomputed_squared_error"] = predictions["recomputed_error"] ** 2

print("\nMETRICS RECOMPUTED FROM ACTUAL AND PREDICTED")
recomputed = (
    predictions.groupby("model")
    .agg(
        n=("recomputed_error", "count"),
        MAE=("recomputed_abs_error", "mean"),
        MSE=("recomputed_squared_error", "mean"),
        Bias=("recomputed_error", "mean"),
    )
    .reset_index()
)
recomputed["RMSE"] = np.sqrt(recomputed["MSE"])
display(
    recomputed.drop(columns="MSE").sort_values("MAE").round(4)
)


ERROR CONVENTION CHECK
Max difference vs (actual - predicted): 64.94591454244724
Max difference vs (predicted - actual): 0.0

DERIVED-COLUMN CHECKS
Max abs_error mismatch: 0.0
Max squared_error mismatch: 0.0

ROWS MATCHING NEITHER ERROR CONVENTION
Count: 0


,fold,model,row_idx,actual,predicted,error,abs_error,squared_error



METRICS RECOMPUTED FROM ACTUAL AND PREDICTED


,model,n,MAE,Bias,RMSE
4,XGBoost,631,3.1908,-0.0455,4.6941
1,Random Forest,631,3.2409,-0.7170,4.4012
2,Ridge Regression,631,3.9549,1.5284,6.0616
0,Linear Regression,631,4.3286,2.9687,7.5373
3,Training median baseline,631,5.8476,3.3989,7.2466
